# Risk Parity Portfolio Strategy

This notebook demonstrates **Risk Parity** portfolio construction as an alternative to mean-variance optimization.

**Core Principle**: Allocate capital so each asset contributes equally to portfolio risk, rather than optimizing expected return.

**Key Concepts**:
1. **Inverse Volatility Weighting** - Simplest form (Naive Risk Parity)
2. **Equal Risk Contribution (ERC)** - Each asset contributes 1/N of total portfolio risk
3. **Correlation Adjustment** - Account for diversification benefits
4. **Leverage** - Scale low-volatility portfolios to target return
5. **Rebalancing** - Drift-based vs calendar-based triggers

**Why Risk Parity?**
- Mean-variance is sensitive to return forecasts (alphas)
- Risk parity depends only on covariance (more stable)
- Suitable when alphas are weak or unreliable
- Historically used by institutional investors (Bridgewater All Weather)

**References**:
- Grinold & Kahn (1999): Active Portfolio Management
- Qian (2005): Risk Parity Portfolios
- Maillard et al. (2010): Equal Risk Contribution Portfolio
- Recent research (2025): Hierarchical Risk Parity

---

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Standard imports
import numpy as np
import pandas as pd
import polars as pl
from datetime import date, timedelta
from typing import List, Dict, Tuple
from scipy.optimize import minimize

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Setup complete!")

---

## Generate Multi-Asset Class Data

We'll create synthetic data for 6 asset classes with different volatilities and correlations:
- **Equities** (high vol, high return)
- **Corporate Bonds** (medium vol, medium return)
- **Government Bonds** (low vol, low return)
- **Real Estate** (medium vol, medium return)
- **Commodities** (high vol, diversifying)
- **Cash** (very low vol, low return)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Define asset classes
asset_classes = ['Equities', 'Corp_Bonds', 'Govt_Bonds', 'Real_Estate', 'Commodities', 'Cash']

# Asset parameters (annualized)
asset_params = {
    'Equities': {'mu': 0.10, 'sigma': 0.18},
    'Corp_Bonds': {'mu': 0.05, 'sigma': 0.08},
    'Govt_Bonds': {'mu': 0.03, 'sigma': 0.04},
    'Real_Estate': {'mu': 0.07, 'sigma': 0.12},
    'Commodities': {'mu': 0.06, 'sigma': 0.20},
    'Cash': {'mu': 0.02, 'sigma': 0.01}
}

# Correlation matrix (designed to reflect realistic relationships)
corr_matrix_data = np.array([
    [1.00, 0.60, 0.20, 0.50, 0.30, 0.05],  # Equities
    [0.60, 1.00, 0.50, 0.40, 0.20, 0.10],  # Corp Bonds
    [0.20, 0.50, 1.00, 0.10, -0.10, 0.15],  # Govt Bonds (flight to safety)
    [0.50, 0.40, 0.10, 1.00, 0.25, 0.05],  # Real Estate
    [0.30, 0.20, -0.10, 0.25, 1.00, 0.00],  # Commodities (inflation hedge)
    [0.05, 0.10, 0.15, 0.05, 0.00, 1.00]   # Cash
])

# Generate 1008 days of data (4 years, ~252 trading days/year)
n_days = 1008
dates = [date(2020, 1, 1) + timedelta(days=i) for i in range(n_days)]

# Generate correlated returns
def generate_correlated_returns(asset_params, corr_matrix, n_days):
    """Generate multivariate normal returns with specified correlation."""
    n_assets = len(asset_params)
    
    # Build covariance matrix from correlation and volatilities
    sigmas = np.array([asset_params[asset]['sigma'] for asset in asset_classes])
    cov_matrix = np.outer(sigmas, sigmas) * corr_matrix
    
    # Generate returns (daily)
    daily_means = np.array([asset_params[asset]['mu'] / 252 for asset in asset_classes])
    daily_cov = cov_matrix / 252
    
    returns = np.random.multivariate_normal(daily_means, daily_cov, n_days)
    
    return returns

returns_matrix = generate_correlated_returns(asset_params, corr_matrix_data, n_days)

# Create DataFrame
returns_df = pd.DataFrame(returns_matrix, columns=asset_classes, index=dates)

print(f"✓ Generated {n_days} days of returns for {len(asset_classes)} asset classes")
print(f"\n📊 Annualized Statistics:")
print("=" * 60)
print(f"{'Asset':<15} {'Return':<10} {'Volatility':<12} {'Sharpe':<8}")
print("=" * 60)

for asset in asset_classes:
    annual_return = returns_df[asset].mean() * 252
    annual_vol = returns_df[asset].std() * np.sqrt(252)
    sharpe = annual_return / annual_vol
    print(f"{asset:<15} {annual_return:>8.2%}  {annual_vol:>10.2%}  {sharpe:>6.2f}")

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(returns_df.corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-0.5, vmax=1, square=True, cbar_kws={'label': 'Correlation'})
plt.title('Return Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("  • Govt Bonds negatively correlated with Commodities (inflation hedge)")
print("  • Equities and Corp Bonds moderately correlated (credit risk)")
print("  • Cash has low correlation with everything (diversifier)")

---

## Theory: Risk Parity Principles

### 1. Marginal Risk Contribution

The **marginal risk contribution** (MRC) of asset $i$ is:

$$
MRC_i = \frac{\partial \sigma_p}{\partial w_i} = \frac{(\Sigma w)_i}{\sigma_p}
$$

Where:
- $\sigma_p = \sqrt{w^T \Sigma w}$ is portfolio volatility
- $\Sigma$ is the covariance matrix
- $(\Sigma w)_i$ is the $i$-th element of $\Sigma w$

### 2. Risk Contribution

The **risk contribution** (RC) of asset $i$ is:

$$
RC_i = w_i \times MRC_i = w_i \times \frac{(\Sigma w)_i}{\sigma_p}
$$

**Property**: $\sum_{i=1}^N RC_i = \sigma_p$ (Euler's homogeneous function theorem)

### 3. Equal Risk Contribution (ERC)

**Goal**: Find weights $w$ such that:

$$
RC_1 = RC_2 = \cdots = RC_N = \frac{\sigma_p}{N}
$$

**Optimization Problem**:

$$
\min_w \sum_{i=1}^N \sum_{j=1}^N (RC_i - RC_j)^2 \quad \text{subject to} \quad \sum_{i=1}^N w_i = 1, \; w_i \geq 0
$$

### 4. Naive Risk Parity (Inverse Volatility)

**Simplification**: Assume zero correlation between assets.

Then: $w_i \propto \frac{1}{\sigma_i}$

$$
w_i = \frac{1/\sigma_i}{\sum_{j=1}^N 1/\sigma_j}
$$

**Advantage**: Simple, closed-form solution

**Disadvantage**: Ignores diversification benefits from correlation

---

## Implementation 1: Naive Risk Parity (Inverse Volatility)

In [ ]:
def naive_risk_parity(returns: pd.DataFrame) -> pd.Series:
    """
    Compute inverse volatility weights (Naive Risk Parity).
    
    Formula: w_i = (1/sigma_i) / sum(1/sigma_j)
    
    Args:
        returns: Historical returns DataFrame
    
    Returns:
        Portfolio weights (Series)
    """
    # Calculate volatilities (annualized)
    volatilities = returns.std() * np.sqrt(252)
    
    # Inverse volatility weights
    inv_vol = 1.0 / volatilities
    weights = inv_vol / inv_vol.sum()
    
    return weights

# Calculate Naive Risk Parity weights
weights_naive_rp = naive_risk_parity(returns_df)

print("📊 Naive Risk Parity Weights (Inverse Volatility):")
print("=" * 50)
for asset, weight in weights_naive_rp.items():
    vol = returns_df[asset].std() * np.sqrt(252)
    print(f"{asset:<15} {weight:>8.2%}  (vol = {vol:.2%})")

print(f"\nTotal: {weights_naive_rp.sum():.2%}")

print("\n💡 Interpretation:")
print("  • Low-volatility assets (Cash, Govt Bonds) get higher weights")
print("  • High-volatility assets (Equities, Commodities) get lower weights")
print("  • Goal: Each asset contributes similar risk to portfolio")

In [ ]:
# Visualize Naive Risk Parity weights
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Weights
axes[0].barh(weights_naive_rp.index, weights_naive_rp.values, color='steelblue', alpha=0.7)
axes[0].set_title('Naive Risk Parity Weights', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Weight', fontsize=10)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].grid(True, alpha=0.3, axis='x')

# Volatilities
vols = returns_df.std() * np.sqrt(252)
axes[1].barh(vols.index, vols.values, color='coral', alpha=0.7)
axes[1].set_title('Asset Volatilities (Annualized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Volatility', fontsize=10)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Notice: Weights are inversely proportional to volatility")

---

## Implementation 2: Equal Risk Contribution (ERC)

Now we account for correlations using numerical optimization.

In [ ]:
def equal_risk_contribution(returns: pd.DataFrame, max_iter: int = 1000) -> pd.Series:
    """
    Compute Equal Risk Contribution (ERC) weights using optimization.
    
    Objective: Minimize sum of squared differences in risk contributions.
    
    Args:
        returns: Historical returns DataFrame
        max_iter: Maximum optimization iterations
    
    Returns:
        Portfolio weights (Series)
    """
    # Covariance matrix (annualized)
    cov_matrix = returns.cov() * 252
    n_assets = len(returns.columns)
    
    # Objective: minimize variance of risk contributions
    def risk_contribution_variance(w):
        """Variance of risk contributions."""
        # Portfolio volatility
        portfolio_vol = np.sqrt(w @ cov_matrix @ w)
        
        # Marginal risk contribution: (Sigma * w) / portfolio_vol
        mrc = (cov_matrix @ w) / portfolio_vol
        
        # Risk contribution: w_i * MRC_i
        rc = w * mrc
        
        # Target: equal risk contribution (1/N of total risk)
        target_rc = portfolio_vol / n_assets
        
        # Sum of squared deviations
        return np.sum((rc - target_rc) ** 2)
    
    # Constraints: weights sum to 1
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    
    # Bounds: long-only
    bounds = [(0.0, 1.0) for _ in range(n_assets)]
    
    # Initial guess: equal weights
    w0 = np.ones(n_assets) / n_assets
    
    # Optimize
    result = minimize(
        risk_contribution_variance,
        w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': max_iter, 'ftol': 1e-9}
    )
    
    if not result.success:
        print(f"⚠️  Optimization did not converge: {result.message}")
    
    weights = pd.Series(result.x, index=returns.columns)
    return weights

# Calculate ERC weights
weights_erc = equal_risk_contribution(returns_df)

print("📊 Equal Risk Contribution (ERC) Weights:")
print("=" * 50)
for asset, weight in weights_erc.items():
    print(f"{asset:<15} {weight:>8.2%}")

print(f"\nTotal: {weights_erc.sum():.2%}")

In [ ]:
# Verify equal risk contributions
def calculate_risk_contributions(weights: pd.Series, returns: pd.DataFrame) -> pd.Series:
    """Calculate risk contribution of each asset."""
    cov_matrix = returns.cov() * 252
    w = weights.values
    
    # Portfolio volatility
    portfolio_vol = np.sqrt(w @ cov_matrix @ w)
    
    # Marginal risk contribution
    mrc = (cov_matrix @ w) / portfolio_vol
    
    # Risk contribution
    rc = w * mrc
    
    return pd.Series(rc, index=weights.index)

# Calculate risk contributions for ERC
rc_erc = calculate_risk_contributions(weights_erc, returns_df)

print("📊 Risk Contributions (ERC):")
print("=" * 60)
print(f"{'Asset':<15} {'Weight':<10} {'Risk Contrib':<15} {'% of Total'}")
print("=" * 60)

total_rc = rc_erc.sum()
for asset in weights_erc.index:
    weight = weights_erc[asset]
    rc = rc_erc[asset]
    pct = rc / total_rc
    print(f"{asset:<15} {weight:>8.2%}  {rc:>12.4f}  {pct:>10.2%}")

print(f"\nTotal Risk: {total_rc:.4f}")
print(f"Std Dev of RC: {rc_erc.std():.6f}  (should be close to 0)")

print("\n💡 Verification:")
print("  • Each asset contributes ~16.7% (1/6) of total portfolio risk")
print("  • Standard deviation of risk contributions is near zero")

---

## Comparison: ERC vs Naive Risk Parity

In [ ]:
# Calculate risk contributions for Naive RP
rc_naive = calculate_risk_contributions(weights_naive_rp, returns_df)

# Compare weights
comparison_df = pd.DataFrame({
    'Naive_RP': weights_naive_rp,
    'ERC': weights_erc,
    'Difference': weights_erc - weights_naive_rp
})

print("📊 Weight Comparison: Naive RP vs ERC")
print("=" * 60)
print(comparison_df.to_string(float_format=lambda x: f'{x:.2%}'))

# Compare risk contributions
rc_comparison_df = pd.DataFrame({
    'Naive_RP': rc_naive / rc_naive.sum(),
    'ERC': rc_erc / rc_erc.sum(),
    'Target': 1.0 / len(asset_classes)
})

print("\n📊 Risk Contribution % Comparison")
print("=" * 60)
print(rc_comparison_df.to_string(float_format=lambda x: f'{x:.2%}'))

print("\n💡 Key Insight:")
print("  • ERC achieves more equal risk contributions by accounting for correlations")
print("  • Naive RP overweights assets with low correlation to others")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Weights comparison
x = np.arange(len(asset_classes))
width = 0.35

axes[0].bar(x - width/2, weights_naive_rp.values, width, label='Naive RP', alpha=0.7, color='steelblue')
axes[0].bar(x + width/2, weights_erc.values, width, label='ERC', alpha=0.7, color='green')
axes[0].set_xlabel('Asset Class', fontsize=10)
axes[0].set_ylabel('Weight', fontsize=10)
axes[0].set_title('Portfolio Weights Comparison', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(asset_classes, rotation=45, ha='right')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Risk contributions comparison
target_rc = np.ones(len(asset_classes)) / len(asset_classes)

axes[1].bar(x - width/2, (rc_naive / rc_naive.sum()).values, width, label='Naive RP', alpha=0.7, color='steelblue')
axes[1].bar(x + width/2, (rc_erc / rc_erc.sum()).values, width, label='ERC', alpha=0.7, color='green')
axes[1].axhline(y=1/len(asset_classes), color='red', linestyle='--', linewidth=2, label='Target (Equal)')
axes[1].set_xlabel('Asset Class', fontsize=10)
axes[1].set_ylabel('Risk Contribution', fontsize=10)
axes[1].set_title('Risk Contribution Comparison', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(asset_classes, rotation=45, ha='right')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

## Leverage Calculation

Risk parity portfolios are often low-volatility (due to high bond allocation). 
We can apply leverage to reach a target volatility or return.

In [ ]:
def calculate_portfolio_stats(weights: pd.Series, returns: pd.DataFrame) -> Dict:
    """
    Calculate portfolio statistics.
    
    Args:
        weights: Portfolio weights
        returns: Historical returns
    
    Returns:
        Dictionary with return, volatility, Sharpe
    """
    # Portfolio returns
    portfolio_returns = (returns * weights).sum(axis=1)
    
    # Annualized statistics
    annual_return = portfolio_returns.mean() * 252
    annual_vol = portfolio_returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol > 0 else 0.0
    
    return {
        'return': annual_return,
        'volatility': annual_vol,
        'sharpe': sharpe,
        'returns_series': portfolio_returns
    }

# Calculate unlevered portfolio stats
stats_naive = calculate_portfolio_stats(weights_naive_rp, returns_df)
stats_erc = calculate_portfolio_stats(weights_erc, returns_df)

print("📊 Unlevered Portfolio Statistics:")
print("=" * 60)
print(f"{'Strategy':<15} {'Return':<12} {'Volatility':<12} {'Sharpe'}")
print("=" * 60)
print(f"{'Naive RP':<15} {stats_naive['return']:>10.2%}  {stats_naive['volatility']:>10.2%}  {stats_naive['sharpe']:>6.2f}")
print(f"{'ERC':<15} {stats_erc['return']:>10.2%}  {stats_erc['volatility']:>10.2%}  {stats_erc['sharpe']:>6.2f}")

# Calculate leverage needed to reach target volatility (e.g., 10%)
target_vol = 0.10

leverage_naive = target_vol / stats_naive['volatility']
leverage_erc = target_vol / stats_erc['volatility']

print(f"\n📊 Leverage to Reach {target_vol:.0%} Target Volatility:")
print("=" * 60)
print(f"Naive RP: {leverage_naive:.2f}x leverage")
print(f"ERC:      {leverage_erc:.2f}x leverage")

# Levered portfolio stats
levered_return_naive = stats_naive['return'] * leverage_naive
levered_return_erc = stats_erc['return'] * leverage_erc

print(f"\n📊 Levered Portfolio Statistics ({target_vol:.0%} vol):")
print("=" * 60)
print(f"{'Strategy':<15} {'Return':<12} {'Volatility':<12} {'Sharpe'}")
print("=" * 60)
print(f"{'Naive RP':<15} {levered_return_naive:>10.2%}  {target_vol:>10.2%}  {stats_naive['sharpe']:>6.2f}")
print(f"{'ERC':<15} {levered_return_erc:>10.2%}  {target_vol:>10.2%}  {stats_erc['sharpe']:>6.2f}")

print("\n💡 Key Insight:")
print("  • Risk parity portfolios are naturally low-vol (high bond allocation)")
print("  • Leverage scales return proportionally while maintaining Sharpe ratio")
print("  • Common practice: lever to match equity-like volatility (~15%)")

---

## Comparison: Equal Weight vs Mean-Variance vs Risk Parity

In [ ]:
# Equal weight portfolioweights_equal = pd.Series(1.0 / len(asset_classes), index=asset_classes)stats_equal = calculate_portfolio_stats(weights_equal, returns_df)# Mean-variance portfolio (maximum Sharpe ratio)# Use existing MeanVarianceOptimizer if available, otherwise simple optimizationfrom Risk.Covariance.LedoitWolfShrinkage import LedoitWolfShrinkagedef maximum_sharpe_portfolio(returns: pd.DataFrame) -> pd.Series:    """Find portfolio with maximum Sharpe ratio."""    # Expected returns (sample mean)    mu = returns.mean() * 252        # Covariance matrix    cov_matrix = returns.cov() * 252        n_assets = len(returns.columns)        # Objective: minimize negative Sharpe ratio    def neg_sharpe(w):        portfolio_return = w @ mu        portfolio_vol = np.sqrt(w @ cov_matrix @ w)        return -portfolio_return / portfolio_vol if portfolio_vol > 0 else 1e6        # Constraints    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]    bounds = [(0.0, 1.0) for _ in range(n_assets)]    w0 = np.ones(n_assets) / n_assets        result = minimize(neg_sharpe, w0, method='SLSQP', bounds=bounds, constraints=constraints)        return pd.Series(result.x, index=returns.columns)weights_mv = maximum_sharpe_portfolio(returns_df)stats_mv = calculate_portfolio_stats(weights_mv, returns_df)# Comparison tableprint("📊 Portfolio Strategy Comparison:")print("=" * 70)print(f"{'Strategy':<20} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")print("=" * 70)print(f"{'Equal Weight':<20} {stats_equal['return']:>10.2%}  {stats_equal['volatility']:>10.2%}  {stats_equal['sharpe']:>8.2f}")print(f"{'Mean-Variance':<20} {stats_mv['return']:>10.2%}  {stats_mv['volatility']:>10.2%}  {stats_mv['sharpe']:>8.2f}")print(f"{'Naive Risk Parity':<20} {stats_naive['return']:>10.2%}  {stats_naive['volatility']:>10.2%}  {stats_naive['sharpe']:>8.2f}")print(f"{'ERC Risk Parity':<20} {stats_erc['return']:>10.2%}  {stats_erc['volatility']:>10.2%}  {stats_erc['sharpe']:>8.2f}")print("\n📊 Portfolio Weights:")print("=" * 80)weights_comparison = pd.DataFrame({    'Equal': weights_equal,    'Mean-Var': weights_mv,    'Naive_RP': weights_naive_rp,    'ERC': weights_erc})print(weights_comparison.to_string(float_format=lambda x: f'{x:.2%}'))

In [ ]:
# Visualize weight allocations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

strategies = ['Equal', 'Mean-Var', 'Naive_RP', 'ERC']
colors = ['steelblue', 'coral', 'green', 'purple']

for idx, (strategy, color) in enumerate(zip(strategies, colors)):
    ax = axes[idx // 2, idx % 2]
    weights = weights_comparison[strategy]
    
    ax.barh(weights.index, weights.values, color=color, alpha=0.7)
    ax.set_title(f'{strategy} Portfolio', fontsize=12, fontweight='bold')
    ax.set_xlabel('Weight', fontsize=10)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("  • Equal Weight: No optimization, equal allocation")
print("  • Mean-Variance: Concentrates in high Sharpe assets (may be unstable)")
print("  • Naive RP: Diversified, but ignores correlations")
print("  • ERC: Best diversification, accounts for correlations")

---

## Rebalancing: Drift vs Calendar

Risk parity portfolios need periodic rebalancing to maintain equal risk contributions.

In [ ]:
def simulate_portfolio_with_rebalancing(
    returns: pd.DataFrame,
    initial_weights: pd.Series,
    rebalance_method: str = 'monthly',
    drift_threshold: float = 0.05
) -> Tuple[pd.Series, int]:
    """
    Simulate portfolio with rebalancing.
    
    Args:
        returns: Historical returns
        initial_weights: Starting portfolio weights
        rebalance_method: 'monthly', 'quarterly', or 'drift'
        drift_threshold: Rebalance if any weight drifts > threshold
    
    Returns:
        (cumulative_returns, num_rebalances)
    """
    dates = returns.index
    weights = initial_weights.copy()
    portfolio_value = 1.0
    portfolio_values = []
    num_rebalances = 0
    
    # Rebalancing schedule
    last_rebalance_date = dates[0]
    
    for i, date in enumerate(dates):
        # Daily return
        daily_return = (returns.loc[date] * weights).sum()
        portfolio_value *= (1 + daily_return)
        portfolio_values.append(portfolio_value)
        
        # Update weights due to price changes (drift)
        weights = weights * (1 + returns.loc[date])
        weights = weights / weights.sum()  # Normalize
        
        # Check if rebalancing needed
        rebalance = False
        
        if rebalance_method == 'monthly':
            # Rebalance at start of each month
            if date.month != last_rebalance_date.month:
                rebalance = True
        elif rebalance_method == 'quarterly':
            # Rebalance at start of each quarter
            if (date.month - 1) // 3 != (last_rebalance_date.month - 1) // 3:
                rebalance = True
        elif rebalance_method == 'drift':
            # Rebalance if any weight drifts too much
            max_drift = np.abs(weights - initial_weights).max()
            if max_drift > drift_threshold:
                rebalance = True
        
        # Rebalance if needed
        if rebalance:
            weights = initial_weights.copy()
            num_rebalances += 1
            last_rebalance_date = date
    
    cumulative_returns = pd.Series(portfolio_values, index=dates)
    return cumulative_returns, num_rebalances

# Simulate different rebalancing frequencies for ERC
cum_returns_monthly, n_rebal_monthly = simulate_portfolio_with_rebalancing(
    returns_df, weights_erc, rebalance_method='monthly'
)
cum_returns_quarterly, n_rebal_quarterly = simulate_portfolio_with_rebalancing(
    returns_df, weights_erc, rebalance_method='quarterly'
)
cum_returns_drift, n_rebal_drift = simulate_portfolio_with_rebalancing(
    returns_df, weights_erc, rebalance_method='drift', drift_threshold=0.05
)

print("📊 Rebalancing Comparison (ERC Portfolio):")
print("=" * 70)
print(f"{'Method':<15} {'# Rebalances':<15} {'Final Value':<15} {'CAGR'}")
print("=" * 70)

for method, cum_ret, n_rebal in [
    ('Monthly', cum_returns_monthly, n_rebal_monthly),
    ('Quarterly', cum_returns_quarterly, n_rebal_quarterly),
    ('Drift (5%)', cum_returns_drift, n_rebal_drift)
]:
    final_value = cum_ret.iloc[-1]
    n_years = len(cum_ret) / 252
    cagr = (final_value ** (1 / n_years)) - 1
    
    print(f"{method:<15} {n_rebal:<15} {final_value:<15.3f} {cagr:>6.2%}")

print("\n💡 Key Insight:")
print("  • More frequent rebalancing = higher returns (for mean-reverting portfolios)")
print("  • But also = higher transaction costs (not modeled here)")
print("  • Drift-based rebalancing is adaptive and often optimal")

In [ ]:
# Visualize cumulative returns
plt.figure(figsize=(14, 6))

plt.plot(cum_returns_monthly.index, cum_returns_monthly.values, label='Monthly Rebalance', linewidth=2)
plt.plot(cum_returns_quarterly.index, cum_returns_quarterly.values, label='Quarterly Rebalance', linewidth=2)
plt.plot(cum_returns_drift.index, cum_returns_drift.values, label='Drift-Based (5%)', linewidth=2)

plt.title('ERC Portfolio: Rebalancing Method Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Portfolio Value', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Performance in Different Correlation Regimes

Risk parity shines when correlations are stable. Let's test in high vs low correlation periods.

In [ ]:
# Calculate rolling correlation (average pairwise correlation)
def rolling_avg_correlation(returns: pd.DataFrame, window: int = 60) -> pd.Series:
    """Calculate rolling average correlation."""
    avg_corrs = []
    dates = []
    
    for i in range(window, len(returns)):
        window_returns = returns.iloc[i-window:i]
        corr_matrix = window_returns.corr()
        
        # Average of off-diagonal elements
        n = len(corr_matrix)
        avg_corr = (corr_matrix.sum().sum() - n) / (n * (n - 1))
        
        avg_corrs.append(avg_corr)
        dates.append(returns.index[i])
    
    return pd.Series(avg_corrs, index=dates)

# Calculate rolling correlation
rolling_corr = rolling_avg_correlation(returns_df, window=60)

# Identify high and low correlation regimes
median_corr = rolling_corr.median()
high_corr_mask = rolling_corr > median_corr
low_corr_mask = rolling_corr <= median_corr

print("📊 Correlation Regimes:")
print("=" * 50)
print(f"Median Correlation: {median_corr:.3f}")
print(f"High Correlation Periods: {high_corr_mask.sum()} days")
print(f"Low Correlation Periods: {low_corr_mask.sum()} days")

# Calculate portfolio returns for each strategy
def calculate_returns_by_regime(
    returns: pd.DataFrame,
    weights: pd.Series,
    regime_mask: pd.Series
) -> Dict:
    """Calculate statistics for specific regime."""
    portfolio_returns = (returns * weights).sum(axis=1)
    
    # Filter to regime dates
    regime_returns = portfolio_returns[regime_mask]
    
    if len(regime_returns) == 0:
        return {'return': 0.0, 'volatility': 0.0, 'sharpe': 0.0}
    
    annual_return = regime_returns.mean() * 252
    annual_vol = regime_returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol > 0 else 0.0
    
    return {'return': annual_return, 'volatility': annual_vol, 'sharpe': sharpe}

# Compare strategies across regimes
strategies = {
    'Equal Weight': weights_equal,
    'Mean-Variance': weights_mv,
    'ERC': weights_erc
}

print("\n📊 Performance by Correlation Regime:")
print("=" * 80)

for regime_name, mask in [('Low Correlation', low_corr_mask), ('High Correlation', high_corr_mask)]:
    print(f"\n{regime_name} Regime:")
    print("-" * 80)
    print(f"{'Strategy':<20} {'Return':<12} {'Volatility':<12} {'Sharpe'}")
    print("-" * 80)
    
    for strategy_name, weights in strategies.items():
        stats = calculate_returns_by_regime(returns_df, weights, mask)
        print(f"{strategy_name:<20} {stats['return']:>10.2%}  {stats['volatility']:>10.2%}  {stats['sharpe']:>6.2f}")

In [ ]:
# Visualize rolling correlation over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Rolling correlation
axes[0].plot(rolling_corr.index, rolling_corr.values, linewidth=2, color='steelblue')
axes[0].axhline(y=median_corr, color='red', linestyle='--', linewidth=2, label=f'Median: {median_corr:.3f}')
axes[0].fill_between(rolling_corr.index, 0, 1, where=high_corr_mask, alpha=0.2, color='red', label='High Correlation')
axes[0].fill_between(rolling_corr.index, 0, 1, where=low_corr_mask, alpha=0.2, color='green', label='Low Correlation')
axes[0].set_ylabel('Avg Correlation', fontsize=10)
axes[0].set_title('Rolling Average Correlation (60-day window)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Portfolio returns
erc_returns = (returns_df * weights_erc).sum(axis=1)
cum_returns_erc = (1 + erc_returns).cumprod()

axes[1].plot(cum_returns_erc.index, cum_returns_erc.values, linewidth=2, color='purple', label='ERC Portfolio')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Cumulative Return', fontsize=10)
axes[1].set_title('ERC Portfolio Cumulative Returns', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Insight:")
print("  • Risk parity performs well in both regimes")
print("  • Mean-variance can suffer in high correlation (less diversification)")
print("  • ERC adapts better to changing correlations")

---

## Drawdown Comparison

One of risk parity's key benefits is reduced drawdowns due to diversification.

In [ ]:
def calculate_drawdown(returns: pd.Series) -> pd.DataFrame:
    """
    Calculate drawdown series.
    
    Args:
        returns: Daily returns
    
    Returns:
        DataFrame with cumulative returns, running max, drawdown
    """
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = (cum_returns - running_max) / running_max
    
    return pd.DataFrame({
        'cumulative': cum_returns,
        'running_max': running_max,
        'drawdown': drawdown
    })

# Calculate drawdowns for each strategy
drawdowns = {}
for strategy_name, weights in strategies.items():
    portfolio_returns = (returns_df * weights).sum(axis=1)
    drawdowns[strategy_name] = calculate_drawdown(portfolio_returns)

# Summary statistics
print("📊 Drawdown Statistics:")
print("=" * 70)
print(f"{'Strategy':<20} {'Max Drawdown':<15} {'Avg Drawdown':<15} {'Recovery Days'}")
print("=" * 70)

for strategy_name, dd_df in drawdowns.items():
    max_dd = dd_df['drawdown'].min()
    avg_dd = dd_df['drawdown'][dd_df['drawdown'] < 0].mean()
    
    # Calculate average recovery time (days from trough to recovery)
    in_drawdown = dd_df['drawdown'] < -0.01
    recovery_days = in_drawdown.sum() / max(1, (~in_drawdown).sum())
    
    print(f"{strategy_name:<20} {max_dd:>13.2%}  {avg_dd:>13.2%}  {recovery_days:>12.1f}")

print("\n💡 Key Insight:")
print("  • Risk parity (ERC) has lower maximum drawdown")
print("  • Drawdowns are shorter and less severe")
print("  • Important for investors with low risk tolerance")

In [ ]:
# Visualize drawdowns
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

colors = ['steelblue', 'coral', 'green']

for idx, (strategy_name, dd_df) in enumerate(drawdowns.items()):
    # Cumulative returns
    axes[idx].plot(dd_df.index, dd_df['cumulative'].values, 
                   linewidth=2, color=colors[idx], label='Portfolio Value')
    axes[idx].plot(dd_df.index, dd_df['running_max'].values, 
                   linewidth=1, linestyle='--', color='gray', label='Previous Peak')
    
    # Fill drawdown area
    axes[idx].fill_between(dd_df.index, dd_df['cumulative'].values, 
                           dd_df['running_max'].values, 
                           alpha=0.3, color='red')
    
    axes[idx].set_ylabel('Portfolio Value', fontsize=10)
    axes[idx].set_title(f'{strategy_name} Portfolio', fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=9, loc='upper left')
    axes[idx].grid(True, alpha=0.3)
    
    # Add max drawdown annotation
    max_dd_idx = dd_df['drawdown'].idxmin()
    max_dd_value = dd_df['drawdown'].min()
    axes[idx].annotate(f'Max DD: {max_dd_value:.2%}',
                       xy=(max_dd_idx, dd_df.loc[max_dd_idx, 'cumulative']),
                       xytext=(10, -30), textcoords='offset points',
                       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7),
                       arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

axes[2].set_xlabel('Date', fontsize=12)
plt.tight_layout()
plt.show()

print("\nRed shaded areas represent drawdowns (portfolio below previous peak)")

---

## Summary: When to Use Risk Parity

### Advantages of Risk Parity:

1. **Stability**: Depends only on covariance, not expected returns (alphas)
2. **Diversification**: Ensures all assets contribute equally to risk
3. **Drawdown Control**: Lower maximum drawdowns vs concentrated portfolios
4. **Regime Resilience**: Performs consistently across correlation regimes

### When to Use Risk Parity:

- ✅ **Weak or unreliable alpha signals** - When return forecasts are noisy
- ✅ **Long-term investing** - When stability matters more than maximizing return
- ✅ **Multi-asset portfolios** - When assets have very different volatilities
- ✅ **Risk-averse investors** - When drawdown control is priority

### When to Use Mean-Variance Instead:

- ✅ **Strong alpha signals** - When you have reliable return forecasts
- ✅ **Tactical trading** - When you want to tilt toward attractive assets
- ✅ **Similar volatilities** - When all assets have comparable risk
- ✅ **Constrained leverage** - When you can't lever low-vol portfolios

### Implementation Recommendations:

1. **Start with ERC**: More sophisticated than naive risk parity
2. **Rebalance quarterly**: Balance between maintenance and transaction costs
3. **Consider drift triggers**: Rebalance when weights drift >5% from target
4. **Apply leverage judiciously**: Scale to target volatility, not target return
5. **Monitor correlations**: Risk parity assumptions break down in crisis (all correlations → 1)

### Extensions (Not Implemented Here):

- **Hierarchical Risk Parity (HRP)**: Use clustering to improve ERC
- **Risk Budgeting**: Allow unequal risk contributions by asset class
- **Dynamic Risk Parity**: Adjust weights based on vol regime
- **Transaction Costs**: Model costs and optimize rebalancing frequency

---

## Connection to ARBS Framework

This risk parity implementation can be integrated with ARBS:

1. **Replace MeanVarianceOptimizer** with `RiskParityOptimizer`
2. **Use when IC is low** (< 0.03) or alphas are unreliable
3. **Combine with signals**: Use risk parity as baseline, tilt with alphas
4. **Futures/Swaps application**: Especially useful for cross-asset carry strategies

Example:
```python
# In MinimalBacktest or StrategyFactory
if alpha_quality == 'weak':
    optimizer = RiskParityOptimizer(method='erc')
else:
    optimizer = MeanVarianceOptimizer(risk_aversion=1.0)
```

---

## Full ARBS Integration

Integrate risk parity with ARBS backtest framework.

In [ ]:
# Import ARBS framework componentsfrom Signals.Base.BaseSignal import BaseSignalfrom Signals.AlphaGenerator import AlphaGeneratorfrom Risk.Covariance.LedoitWolfShrinkage import LedoitWolfShrinkagefrom Optimizer.MeanVarianceOptimizer import MeanVarianceOptimizerfrom Backtest.MinimalBacktest import MinimalBacktestfrom Analysis.TearSheet import TearSheetfrom Risk.Volatility.RealizedVolatility import RealizedVolatilityprint("✓ ARBS components imported")

In [ ]:
# Create RiskParitySignal that extends BaseSignalclass RiskParitySignal(BaseSignal):    """    Risk Parity signal that uses equal risk contribution.        Instead of forecasting returns, this signal uses inverse volatility    as a proxy for 'attractiveness' - lower vol assets get higher signals.    """        def __init__(self, lookback: int = 60):        self.lookback = lookback        def generate(self, returns_df: pl.DataFrame) -> pl.DataFrame:        """        Generate signals based on inverse volatility.                Returns:            Polars DataFrame with columns [date, ticker, signal]            where signal = 1 / volatility (normalized)        """        # Convert to pandas for easier manipulation        returns_pd = returns_df.pivot(            index='date',            columns='ticker',            values='return'        ).to_pandas()                # Calculate rolling volatility        rolling_vol = returns_pd.rolling(window=self.lookback).std() * np.sqrt(252)                # Inverse volatility signals        inv_vol = 1.0 / rolling_vol                # Normalize signals        signals = inv_vol.div(inv_vol.sum(axis=1), axis=0)                # Convert back to long format        signals_data = []        for date in signals.index:            if pd.isna(signals.loc[date]).any():                continue            for ticker in signals.columns:                signals_data.append({                    'date': date,                    'ticker': ticker,                    'signal': signals.loc[date, ticker]                })                return pl.DataFrame(signals_data)# Generate risk parity signalsrp_signal = RiskParitySignal(lookback=60)# Convert returns_df to Polars format if neededreturns_pl = pl.DataFrame({    'date': returns_df.index.tolist() * len(asset_classes),    'ticker': [asset for asset in asset_classes for _ in range(len(returns_df))],    'return': returns_df.values.flatten()})signals_df = rp_signal.generate(returns_pl)print(f"✓ Generated {len(signals_df)} risk parity signal observations")

In [ ]:
# Configure ARBS pipelinealpha_generator = AlphaGenerator(    IC=0.03,  # Low IC - risk parity doesn't forecast returns    vol_estimator=RealizedVolatility(lookback=60, annualization_factor=252))cov_estimator = LedoitWolfShrinkage()optimizer = MeanVarianceOptimizer(    risk_aversion=3.0,    long_only=True,    leverage_limit=1.0)print("✓ ARBS pipeline configured")# Run backtestprint("\n🚀 Running risk parity backtest...")backtest = MinimalBacktest(    signals_df=signals_df,    returns_df=returns_pl,    alpha_generator=alpha_generator,    cov_estimator=cov_estimator,    optimizer=optimizer)result = backtest.run()print(f"\n✓ Backtest complete!")print(f"Total Return: {result.total_return:.2%}")print(f"Sharpe Ratio: {result.sharpe_ratio:.3f}")print(f"Information Coefficient: {result.IC:.4f}")

In [ ]:
# Generate comprehensive analysistearsheet = TearSheet(    returns=result.returns,    signals_df=signals_df,    returns_df=returns_pl)print("\n" + "="*80)print("RISK PARITY STRATEGY - FULL PERFORMANCE ANALYSIS")print("="*80)tearsheet.plot_all()print("\n✅ ARBS Integration Complete!")print("\nKey Points:")print("  • RiskParitySignal extends BaseSignal")print("  • Uses inverse volatility as signal strength")print("  • Full pipeline: Signal → Alpha → Optimizer → Backtest → TearSheet")print("  • Low IC is expected - risk parity doesn't forecast returns")print("  • Focus is on risk-adjusted returns (Sharpe), not total return")